# Homestead Model Battle Arena on Colab T4

Sweep several sub-4B-class GGUF models through the **same** five real tool-calling
tasks and produce a pass-rate + latency leaderboard -- all on one free Colab T4, with
no dependencies beyond this repo and a couple of pip packages.

**How the battle works:** each model gets asked the same 5 prompts, each exposing a
small set of Python tools (`get_weather`, `celsius_to_fahrenheit`, `calculator`). A
plain tool-calling loop (below, `run_agent()`) sends the prompt, executes whatever
tool the model calls, feeds the result back, and repeats until the model gives a
final answer or 4 rounds pass. A small structural scorer then checks whether the
*right* tool got called with the *right* argument, not whether the final sentence
merely sounds plausible -- a model that stumbles through three wrong tool calls
before a plausible-sounding answer is a real failure this scorer catches that an
LLM-judge-over-text-output would miss.

**Models in this battle:**

| Model | Params | Why it's here |
|---|---|---|
| `LiquidAI/LFM2.5-350M-GGUF` | 350M | floor of the range -- expect tool-calling to mostly collapse |
| `LiquidAI/LFM2.5-1.2B-Instruct-GGUF` | 1.2B | the notebook 01 baseline |
| `LiquidAI/LFM2.5-2.6B-GGUF` | 2.6B | Liquid's own "best quality/speed tradeoff" pick for agent workloads |
| `LiquidAI/LFM2.5-8B-A1B-GGUF` | 8.3B total / ~1.5B active (MoE) | tests whether MoE sparsity buys back quality without the latency of a dense 8B |
| `XHToken/Spark-X2.5-4B-GGUF` | 4B | a non-Liquid, similarly-sized competitor, as an outside comparison point |

**Colab/T4 caveats:**
- Total download across five models is a few GB -- the real time cost here is
  bandwidth, not compute. Each model finishes all 5 tasks in well under a minute.
- The GPU is freed between models (each model's server process is terminated before
  the next one starts) -- don't run cells out of order or two servers will fight over
  the same T4.
- 5 tasks per model is directional, not statistically rigorous -- read pass rates as
  "which model is clearly stronger," not as a precise percentage.

Series: this is notebook **02**. See [`examples/colab/README.md`](./README.md) for
the full lineup and what 03-07 (not yet built) would add.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


## 1. Clone this repo, load shared helpers, install deps


In [ ]:
import os
import subprocess
import sys

if not os.path.isdir("/content/homestead"):
    subprocess.run(
        ["git", "clone", "--quiet", "--depth", "1",
         "https://github.com/olanigan/homestead.git",
         "/content/homestead"],
        check=True,
    )

subprocess.run(
    ["pip", "install", "-q", "openai", "huggingface_hub", "requests", "pandas", "matplotlib"],
    check=True,
)

sys.path.insert(0, "/content/homestead/examples/colab")
from colab_common import try_build_gateway, serve_model, pick_and_download_gguf


## 2. The tools, tasks, agent loop, and scorer

Everything the battle needs, self-contained in this notebook -- three tiny Python
tools, five tasks (prompt + what a correct run must do), a generic OpenAI-compatible
tool-calling loop, and a structural pass/fail scorer.


In [ ]:
import ast
import json
import operator
import time


def get_weather(city: str) -> str:
    return f"It's sunny and 22C in {city}."


def celsius_to_fahrenheit(celsius: float) -> float:
    return celsius * 9 / 5 + 32


_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}


def calculator(expression: str) -> str:
    def _eval(node):
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        raise ValueError(f"unsupported expression: {expression}")
    return str(_eval(ast.parse(expression, mode="eval").body))


TOOLS = {"get_weather": get_weather, "celsius_to_fahrenheit": celsius_to_fahrenheit, "calculator": calculator}

TOOL_SPECS = {
    "get_weather": {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]},
        },
    },
    "celsius_to_fahrenheit": {
        "type": "function",
        "function": {
            "name": "celsius_to_fahrenheit",
            "description": "Convert a Celsius temperature to Fahrenheit.",
            "parameters": {"type": "object", "properties": {"celsius": {"type": "number"}}, "required": ["celsius"]},
        },
    },
    "calculator": {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a simple arithmetic expression, e.g. '2 + 2'. Supports + - * /.",
            "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]},
        },
    },
}

TASKS = [
    {
        "id": "001-single-tool-call",
        "prompt": "What's the weather in Lagos right now?",
        "tools": ["get_weather"],
        "expect": {"tool_calls": [{"name": "get_weather", "args_contains": {"city": "Lagos"}}], "min_calls": 1, "max_calls": 1},
    },
    {
        "id": "002-tool-selection",
        "prompt": "What is 12 times 4?",
        "tools": ["get_weather", "celsius_to_fahrenheit", "calculator"],
        "expect": {"tool_calls": [{"name": "calculator"}], "min_calls": 1, "max_calls": 2},
    },
    {
        "id": "003-multi-step-chain",
        "prompt": "What's the weather in Lagos, in Fahrenheit?",
        "tools": ["get_weather", "celsius_to_fahrenheit"],
        "expect": {"tool_calls": [{"name": "get_weather"}, {"name": "celsius_to_fahrenheit"}], "ordered": True, "min_calls": 2},
    },
    {
        "id": "004-no-tool-needed",
        "prompt": "In one short sentence, what color is the sky on a clear day?",
        "tools": ["get_weather", "celsius_to_fahrenheit", "calculator"],
        "expect": {"tool_calls": [], "max_calls": 0},
    },
    {
        "id": "005-arg-extraction",
        "prompt": "I'm flying into Lagos tomorrow -- what should I expect weather-wise?",
        "tools": ["get_weather"],
        "expect": {"tool_calls": [{"name": "get_weather", "args_contains": {"city": "Lagos"}}], "min_calls": 1, "max_calls": 1},
    },
]


def run_agent(client, model_id, prompt, tool_names, max_rounds=4):
    messages = [{"role": "user", "content": prompt}]
    tools_spec = [TOOL_SPECS[name] for name in tool_names]
    tool_calls_made = []
    for _ in range(max_rounds):
        resp = client.chat.completions.create(model=model_id, messages=messages, tools=tools_spec)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content or "", tool_calls_made
        messages.append(msg.model_dump())
        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
                result = TOOLS[name](**args) if name in TOOLS else f"error: unknown tool {name}"
            except Exception as e:
                args, result = {}, f"error: {e}"
            tool_calls_made.append({"name": name, "args": args})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
    return "", tool_calls_made  # exceeded max_rounds without a final answer


def score_task(task, tool_calls, error=None):
    expect = task["expect"]
    reasons = []
    ok = True
    if error:
        return {"passed": False, "reasons": [f"run error: {error}"], "num_tool_calls": len(tool_calls)}

    expected_calls = expect.get("tool_calls", [])
    if expected_calls == [] and "tool_calls" in expect:
        if tool_calls:
            ok = False
            reasons.append(f"expected no tool calls, got {[c['name'] for c in tool_calls]}")
    elif expect.get("ordered"):
        actual_names = [c["name"] for c in tool_calls]
        expected_names = [c["name"] for c in expected_calls]
        if actual_names[: len(expected_names)] != expected_names:
            ok = False
            reasons.append(f"expected ordered calls {expected_names}, got {actual_names}")
    else:
        actual_names = {c["name"] for c in tool_calls}
        for call in expected_calls:
            if call["name"] not in actual_names:
                ok = False
                reasons.append(f"missing expected call to {call['name']}")
                continue
            args_contains = call.get("args_contains", {})
            if args_contains:
                matching = [c for c in tool_calls if c["name"] == call["name"]]
                if not any(all(str(v) in str(c["args"]) for v in args_contains.values()) for c in matching):
                    ok = False
                    reasons.append(f"{call['name']} called but args didn't contain {args_contains}")

    if "min_calls" in expect and len(tool_calls) < expect["min_calls"]:
        ok = False
        reasons.append(f"expected >= {expect['min_calls']} tool calls, got {len(tool_calls)}")
    if "max_calls" in expect and len(tool_calls) > expect["max_calls"]:
        ok = False
        reasons.append(f"expected <= {expect['max_calls']} tool calls, got {len(tool_calls)}")

    return {"passed": ok, "reasons": reasons, "num_tool_calls": len(tool_calls)}


print(f"{len(TASKS)} tasks, {len(TOOLS)} tools ready.")


## 3. Build the gateway once (reused across every model in the battle)

Same try-then-verify-then-fallback logic as notebook 01 -- built once here since the
gateway binary itself doesn't change between models, only which model gets served
(and, best-effort, registered) on top of it.


In [ ]:
gateway_ok = try_build_gateway()
print("Gateway build:", "OK" if gateway_ok else "skipped/failed -- each model will fall back to llama.cpp individually if needed")


## 4. The battle: download -> serve -> run all 5 tasks -> tear down, per model


In [ ]:
import re

from openai import OpenAI

MODELS = [
    "LiquidAI/LFM2.5-350M-GGUF",
    "LiquidAI/LFM2.5-1.2B-Instruct-GGUF",
    "LiquidAI/LFM2.5-2.6B-GGUF",
    "LiquidAI/LFM2.5-8B-A1B-GGUF",
    "XHToken/Spark-X2.5-4B-GGUF",
]


def slugify(repo_id):
    return re.sub(r"[^a-zA-Z0-9]+", "-", repo_id).strip("-").lower()


rows = []

for repo_id in MODELS:
    print(f"\n=== {repo_id} ===")
    model_path = pick_and_download_gguf(repo_id)

    proc, base_url, models, used_gateway = serve_model(model_path, gateway_ok)
    if not models:
        print(f"!! {repo_id} never came up healthy -- skipping, check /content/*.log")
        continue
    model_id = models["data"][0]["id"]
    print(f"Serving {repo_id} as '{model_id}' via {'gateway' if used_gateway else 'llama.cpp'} at {base_url}")

    client = OpenAI(base_url=base_url, api_key="none")

    for task in TASKS:
        start = time.perf_counter()
        error = None
        try:
            _, tool_calls = run_agent(client, model_id, task["prompt"], task["tools"])
        except Exception as e:
            tool_calls, error = [], str(e)
        wall_ms = (time.perf_counter() - start) * 1000
        score = score_task(task, tool_calls, error=error)
        print(f"  {task['id']}: {'PASS' if score['passed'] else 'FAIL'} ({wall_ms:.0f}ms)" + (f" -- {score['reasons']}" if not score['passed'] else ""))
        rows.append({
            "model": repo_id, "task_id": task["id"], "passed": score["passed"],
            "num_tool_calls": score["num_tool_calls"], "wall_ms": wall_ms,
        })

    proc.terminate()
    time.sleep(2)  # let the T4 actually free up before the next model claims it

print("\nBattle complete.")


## 5. The leaderboard

`wall_ms` is wall-clock time for the whole round trip (the tool-calling loop, tool
execution, network) -- not isolated model decode speed. Treat it as "how long did
this feel," not a tokens/sec benchmark.


In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
df.to_csv("/content/battle-results.csv", index=False)
df


In [ ]:
pivot = df.pivot(index="task_id", columns="model", values="passed")
# pandas 3.x removed DataFrame.applymap entirely -- .map() is the replacement.
pivot = pivot.map(lambda v: "PASS" if v is True else ("FAIL" if v is False else "-"))
print(pivot.to_string())

summary = (
    df.groupby("model")
    .agg(pass_rate=("passed", "mean"), median_wall_ms=("wall_ms", "median"))
    .sort_values("pass_rate", ascending=False)
)
summary["pass_rate"] = (summary["pass_rate"] * 100).round(0).astype(int).astype(str) + "%"
summary


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

pass_rate_pct = df.groupby("model")["passed"].mean() * 100
pass_rate_pct.sort_values().plot(kind="barh", ax=axes[0], color="#4C72B0")
axes[0].set_title("Pass rate by model (5 tasks)")
axes[0].set_xlabel("% tasks passed")
axes[0].set_xlim(0, 100)

median_latency = df.groupby("model")["wall_ms"].median()
median_latency.sort_values().plot(kind="barh", ax=axes[1], color="#DD8452")
axes[1].set_title("Median round-trip latency by model")
axes[1].set_xlabel("ms")

plt.tight_layout()
plt.savefig("/content/leaderboard.png", dpi=150)
plt.show()


## What's next

- **03 Framework Showdown** (planned) -- fix the winning model from this battle, vary
  the framework (LangGraph / CrewAI / a hand-rolled loop like this one) instead, to
  see how much of the remaining failure rate is framework glue rather than model
  capability.
- **07 David vs. Goliath** (planned) -- take this battle's best small model and put it
  up against a big hosted API model on the same task suite, for a cost/latency/quality
  comparison.
- See [`examples/colab/README.md`](./README.md) for the full roadmap.
